# Обучение модели HistGradientBoosting для задачи бинарной классификации - предсказания купит ли пользователь товар

In [ ]:
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

from purchase_intent.config import settings
from purchase_intent.features import add_features

In [ ]:
SEED = 42
TEST_SIZE = 0.20

DATA_PATH = settings.data_dir / "online_shoppers_intention.csv"
ARTIFACT_PATH = settings.model_path

ARTIFACT_VERSION = "2.0.0"

CLASSIFICATION_THRESHOLD = 0.27

HGB_PARAMS = {
    "learning_rate": 0.05,
    "max_depth": 6,
    "random_state": SEED,
}

## 1. Загрузка данных

In [ ]:
df = pd.read_csv(DATA_PATH)

TARGET = "Revenue"

df = df.drop_duplicates().reset_index(drop=True)

df[TARGET] = df[TARGET].astype(int)
# df["Weekend"] = df["Weekend"].astype(int)

# CODE_CAT_COLS = [
#     "OperatingSystems",
#     "Browser",
#     "Region",
#     "TrafficType",
# ]
# df[CODE_CAT_COLS] = df[CODE_CAT_COLS].astype(str)

X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print(f"Всего объектов после удаления дубликатов: {len(df)}")
print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")
print(f"Доля класса 1: train={y_train.mean():.3f}, test={y_test.mean():.3f}")

## 2. Feature engineering

Функция `add_features` лежит в папке (src/purchase_intent/features.py), а не в ноутбуке: joblib сохраняет функцию как ссылку "модуль + имя", и сервис должен найти ее по этой ссылке при загрузке бандла. Функция также приводит типы: коды категорий в строки, Weekend в 0/1, поэтому данные из CSV и из запроса к сервису проходят одинаковую обработку.

In [ ]:
FE = FunctionTransformer(add_features)

new_cols = [c for c in add_features(X_train.head()).columns if c not in X_train.columns]
add_features(X_train)[new_cols].describe().T.round(3)

## 3. Предобработка

In [ ]:
LOG_COLS = [
    "Administrative",
    "ProductRelated",
    "Administrative_avg_time",
    "Informational_avg_time",
    "ProductRelated_avg_time",
]

NUM_COLS = ["ExitRates"]

CAT_COLS = [
    "Month",
    "VisitorType",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
]

BIN_COLS = [
    "Weekend",
    "has_PageValues",
    "has_Informational",
    "has_SpecialDay",
    "has_BounceRates",
]


def make_preprocessor():
    log_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ]
    )

    num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

    categorical_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "ohe",
                OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    min_frequency=0.01,
                    sparse_output=False,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("log_num", log_pipeline, LOG_COLS),
            ("num", num_pipeline, NUM_COLS),
            ("cat", categorical_pipeline, CAT_COLS),
            ("bin", "passthrough", BIN_COLS),
        ],
        remainder="drop",
    )

## 4. Pipeline и HistGradientBoosting

In [ ]:
pipeline = Pipeline(
    [
        ("fe", FE),
        ("preprocessing", make_preprocessor()),
        ("clf", HistGradientBoostingClassifier(**HGB_PARAMS)),
    ]
)

pipeline

## 5. Обучение

In [ ]:
pipeline.fit(X_train, y_train)

## 6. Финальная проверка на тестовой выборке

In [ ]:
test_proba = pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= CLASSIFICATION_THRESHOLD).astype(int)

test_metrics = {
    "average_precision": average_precision_score(y_test, test_proba),
    "roc_auc": roc_auc_score(y_test, test_proba),
    "f1": f1_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred),
    "recall": recall_score(y_test, test_pred),
    "accuracy": accuracy_score(y_test, test_pred),
}

pd.Series(test_metrics).round(4)

In [ ]:
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["нет покупки", "покупка"],
        digits=4,
    )
)

In [ ]:
cm = confusion_matrix(y_test, test_pred)

print("Матрица ошибок:")
cm

## 7. Metadata

In [ ]:
FEATURES = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

assert FEATURES == list(
    X_train.columns
), "Порядок входных признаков изменился. Проверьте схему исходных данных."

assert (
    pipeline["fe"].func.__module__ == "purchase_intent.features"
), "функция add_features должна импортироваться из пакета, иначе сервис не загрузит бандл."

metadata = {
    "model_version": ARTIFACT_VERSION,
    "features": FEATURES,
    "threshold": CLASSIFICATION_THRESHOLD,
}

metadata

## 8. Сохранение joblib-бандла

In [ ]:
bundle = {
    "pipeline": pipeline,
    "metadata": metadata,
}

ARTIFACT_PATH.parent.mkdir(exist_ok=True)
joblib.dump(bundle, ARTIFACT_PATH)

print(f"Артефакт сохранен: {ARTIFACT_PATH}")